# ⚙️ 1. Setup DuckDB

Remplacement de Spark par DuckDB :
- ✅ Pas de JVM → démarrage instantané
- ✅ ~300 Mo RAM au lieu de 4-6 Go
- ✅ Lit directement les fichiers Delta/Parquet
- ✅ Requêtes < 100ms sur 1M+ lignes

ATTENTION : Il s'agit juste d'un test personnel

In [1]:
# pip install duckdb
import duckdb

# Connexion in-process — pas de serveur à démarrer
# threads = nb de cœurs CPU à utiliser (None = auto-detect)
con = duckdb.connect(database=':memory:', config={
    'threads': 4,
    'memory_limit': '1GB'  # Plafond RAM strict
})

# Extension Delta Lake (lit vos fichiers Delta natifs)
con.execute("INSTALL delta; LOAD delta;")

print('✅ DuckDB ready')

✅ DuckDB ready


# 📥 2. Chargement des données

**Stratégie** : on crée des VIEWs — DuckDB ne charge rien en RAM tant qu'on n'interroge pas.
Les fichiers Parquet/Delta restent sur disque et sont lus par blocs vectorisés à la demande.

In [2]:
DATA_PATH = "../data/recipes_parquets"

# VIEWs paresseuses — aucun scan au moment de cette cellule
con.execute(f"""
    CREATE OR REPLACE VIEW v_main AS
        SELECT * FROM delta_scan('{DATA_PATH}/recipes_main');

    CREATE OR REPLACE VIEW v_index AS
        SELECT * FROM delta_scan('{DATA_PATH}/ingredients_index');

    CREATE OR REPLACE VIEW v_nutrition AS
        SELECT * FROM delta_scan('{DATA_PATH}/recipes_nutrition_detail');
""")

# ⚡ Optionnel mais recommandé : charger recipes_main en RAM une seule fois
# pour les requêtes répétées (si votre machine a 4+ Go de RAM dispo)
con.execute(f"""
    CREATE OR REPLACE TABLE t_main AS SELECT * FROM v_main;
    CREATE OR REPLACE TABLE t_nutrition AS SELECT * FROM v_nutrition;
    CREATE OR REPLACE TABLE t_index AS SELECT * FROM v_index;
""")

count = con.execute("SELECT COUNT(*) FROM t_main").fetchone()[0]
print(f'✅ {count:,} recettes chargées')

✅ 1,029,720 recettes chargées


# 🧠 3. Vue master + index full-text

L'index FTS (Full-Text Search) remplace les `ILIKE '%...%'` qui faisaient
un scan complet. Gain : **10-50x plus rapide** sur les recherches texte.

In [4]:
# Vue master : join fait une seule fois, stocké en table
con.execute("""
    CREATE OR REPLACE TABLE t_master AS
    SELECT
        m.recipe_id,
        m.title,
        m.cook_minutes,
        m.cook_time_category,
        m.nutri_score,
        m.image_url,
        m.ingredients_raw,
        m.instructions_text,
    FROM t_main m
    LEFT JOIN t_nutrition n USING (recipe_id);
""")

# Index Full-Text Search sur title + ingredients_raw
con.execute("INSTALL fts; LOAD fts;")
con.execute("""
    PRAGMA create_fts_index(
        't_master',         -- table
        'recipe_id',        -- clé primaire
        'title',            -- colonnes indexées
        'ingredients_raw',
        stemmer = 'english',
        overwrite = 1
    );
""")

print('✅ t_master créé + index FTS activé')

✅ t_master créé + index FTS activé


# 🔎 4. Recherche par nom

In [5]:
def search_by_name(query: str, limit: int = 20) -> list[dict]:
    """
    Utilise FTS si disponible, sinon ILIKE en fallback.
    Retourne une liste de dicts — pas de toPandas(), pas de collect() complet.
    """
    result = con.execute("""
        SELECT recipe_id, title, cook_minutes, nutri_score, image_url,
               fts_main_t_master.match_bm25(recipe_id, ?, fields := 'title') AS score
        FROM t_master
        WHERE score IS NOT NULL
        ORDER BY score DESC
        LIMIT ?
    """, [query, limit])
    return result.df().to_dict('records')

# Test
results = search_by_name('pasta')
print(f"{len(results)} résultats")
results[:3]

20 résultats


[{'recipe_id': '871bf87d33',
  'title': 'Homemade Pasta without pasta machine',
  'cook_minutes': None,
  'nutri_score': 'E',
  'image_url': 'https://img-global.cpcdn.com/001_recipes/2432172_b63e13030f76528e/0x0/photo.jpg',
  'score': 2.4855526093768803},
 {'recipe_id': '4028c6cda8',
  'title': 'Basic Pasta With Semolina (Eggless Pasta) Recipe',
  'cook_minutes': None,
  'nutri_score': 'C',
  'image_url': 'https://s-media-cache-ak0.pinimg.com/736x/4a/70/34/4a70346d63d30e7ed2a5c2b6050d9b60.jpg',
  'score': 2.4855526093768803},
 {'recipe_id': '89d7142bea',
  'title': 'Basic Pasta With Semolina (Eggless Pasta)',
  'cook_minutes': None,
  'nutri_score': 'E',
  'image_url': 'https://s-media-cache-ak0.pinimg.com/originals/58/75/bc/5875bc184bae8acd50c92fd2617bfc57.jpg',
  'score': 2.4396175983137733}]

# 🥕 5. Recherche par ingrédient

In [6]:
def search_by_ingredient(ingredient: str, limit: int = 20) -> list[dict]:
    """
    FTS sur ingredients_raw — bien plus rapide que ILIKE '%...%' sur 1M lignes.
    """
    result = con.execute("""
        SELECT recipe_id, title, cook_minutes, nutri_score, image_url, ingredients_raw,
               fts_main_t_master.match_bm25(recipe_id, ?, fields := 'ingredients_raw') AS score
        FROM t_master
        WHERE score IS NOT NULL
        ORDER BY score DESC
        LIMIT ?
    """, [ingredient, limit])
    return result.df().to_dict('records')

# Test
results = search_by_ingredient('tomato')
print(f"{len(results)} résultats")
results[:3]

20 résultats


[{'recipe_id': '6ad65bafb3',
  'title': 'Tomatoes',
  'cook_minutes': None,
  'nutri_score': 'E',
  'image_url': 'https://www.organicfacts.net/wp-content/uploads/2013/05/Organictomato1.jpg',
  'ingredients_raw': array(['1 to 2 Italian Roma tomatoes', '1 to 2 beef steak tomatoes',
         '1 to 2 yellow tomatoes'], dtype=object),
  'score': 1.4985599053565017},
 {'recipe_id': 'da73a59967',
  'title': 'Spaghetti',
  'cook_minutes': 25,
  'nutri_score': 'E',
  'image_url': 'https://img-global.cpcdn.com/001_recipes/4852656521609216/0x0/photo.jpg',
  'ingredients_raw': array(['lb turkey berger', 'can diced tomatoes', 'can tomato paste',
         'can fresh or store tomatoes sauce'], dtype=object),
  'score': 1.4680156542686966},
 {'recipe_id': 'c98cd9a6f8',
  'title': 'Pizza Sauce',
  'cook_minutes': 80,
  'nutri_score': 'A',
  'image_url': 'http://cookingwithchrissy.files.wordpress.com/2011/02/086.jpg',
  'ingredients_raw': array(['12 cup onion, chopped', '3 garlic cloves, minced',
      

# 🧠 6. Recherche avancée

In [8]:
def search_advanced(
    name: str = None,
    ingredient: str = None,
    max_time: int = None,
    nutri_score: str = None,
    limit: int = 20
) -> list[dict]:
    """
    Filtres composables avec paramètres liés (pas de f-string → sécurisé contre injection SQL).
    Les filtres numériques/catégoriels sont appliqués EN PREMIER (très sélectifs),
    le FTS en dernier (plus coûteux).
    """
    conditions = ["1=1"]
    params = []

    # Filtres rapides en premier (index ou scan vectorisé)
    if max_time:
        conditions.append("cook_minutes <= ?")
        params.append(max_time)

    if nutri_score:
        conditions.append("nutri_score = ?")
        params.append(nutri_score.upper())

    where_clause = " AND ".join(conditions)

    # FTS optionnel sur name + ingredient
    fts_filter = ""
    if name or ingredient:
        fts_query = " ".join(filter(None, [name, ingredient]))
        fts_filter = f"""
            AND fts_main_t_master.match_bm25(
                recipe_id, ?,
                fields := '{'title' if name and not ingredient else 'ingredients_raw' if ingredient and not name else 'title, ingredients_raw'}'
            ) IS NOT NULL
        """
        params.insert(0, fts_query)  # FTS param en premier pour ORDER BY

    query = f"""
        SELECT recipe_id, title, cook_minutes, cook_time_category,
               nutri_score, image_url, ingredients_raw
        FROM t_master
        WHERE {where_clause} {fts_filter}
        LIMIT ?
    """
    params.append(limit)

    return con.execute(query, params).df().to_dict('records')

# Test
results = search_advanced(max_time=30, nutri_score='A')
print(f"{len(results)} résultats")
results[:2]

20 résultats


[{'recipe_id': '22e5882cfb',
  'title': 'Better Frizzled Cabbage',
  'cook_minutes': 20,
  'cook_time_category': 'rapide',
  'nutri_score': 'A',
  'image_url': 'http://www.annsentitledlife.com/wp-content/uploads/2013/10/frizzled_cabbage.jpg',
  'ingredients_raw': array(['1 (16 ounce) bag coleslaw mix', '2 slices bacon, diced',
         'soy sauce or teriyaki sauce'], dtype=object)},
 {'recipe_id': '0ad9232f41',
  'title': 'Marinade for Steak',
  'cook_minutes': 15,
  'cook_time_category': 'rapide',
  'nutri_score': 'A',
  'image_url': 'http://img.sndimg.com/food/image/upload/w_512,h_512,c_fit,fl_progressive,q_95/v1/img/recipes/83/47/8/picfINMRO.jpg',
  'ingredients_raw': array(['1 tablespoon olive oil', '2 tablespoons soy sauce',
         '2 tablespoons red wine',
         '2 tablespoons tomato sauce (ketchup) or 2 tablespoons barbecue sauce',
         '3 garlic cloves, crushed', '1 teaspoon dried herbs',
         'good grinding black pepper (or crushed chili to taste)'],
        dtype

# 🍽️ 7. Affichage recette complète

In [9]:
def show_recipe(recipe_id: str) -> dict | None:
    """
    Lookup par clé primaire — O(1) si recipe_id est indexé.
    """
    result = con.execute("""
        SELECT * FROM t_master WHERE recipe_id = ? LIMIT 1
    """, [recipe_id]).fetchone()

    if not result:
        print('❌ Recette introuvable')
        return None

    cols = [desc[0] for desc in con.description]
    r = dict(zip(cols, result))

    print('🍽️ ', r.get('title', '—'))
    print('⏱️  Temps      :', r.get('cook_minutes'), 'min  [', r.get('cook_time_category'), ']')
    print('🔥 Calories   :', r.get('energy_kcal'), 'kcal')
    print('🥗 Nutri-score:', r.get('nutri_score'))
    print('🖼️  Image      :', r.get('image_url'))
    print('\n📝 Ingrédients:')
    print(r.get('ingredients_raw', '—'))
    print('\n👨\u200d🍳 Instructions:')
    print(r.get('instructions_text', '—'))
    return r

# Test
first_id = con.execute("SELECT recipe_id FROM t_master LIMIT 1").fetchone()[0]
show_recipe(first_id)

🍽️  Beet & Corn Risotto
⏱️  Temps      : None min  [ inconnu ]
🔥 Calories   : None kcal
🥗 Nutri-score: E
🖼️  Image      : http://tastykitchen.com/recipes/wp-content/uploads/sites/2/2015/10/beet_corn_rissoto_tk-410x273.jpg

📝 Ingrédients:
['3 whole Fresh Beets', '3 Tablespoons Olive Oil, Divided', '3 ears Corn, Kernels Removed (or About 2 Cups)', '5- 1/2 cups Chicken Stock', '2 Tablespoons Butter', '1/2 whole Onion', '2 cloves Garlic', '1- 1/2 cup Arborio Rice', '1 teaspoon Dried Thyme', '2 Tablespoons Lemon Juice, Plus Lemon Slices For Garnish', '2 teaspoons Salt', '1 teaspoon Ground Pepper', '1/2 cups Parmesan Cheese, Grated']

👨‍🍳 Instructions:
Preheat oven to 400 degrees F. Cover a baking sheet with a piece of parchment paper. | Peel the beets with a vegetable peeler, cut into chunks, and place on the parchment-lined baking sheet. | Toss with 1 tablespoon of the olive oil. | Roast beets for 20 minutes, then remove from the oven and add the corn (and lemon slices if desired, for garn

{'recipe_id': '64f5a5a2c7',
 'title': 'Beet & Corn Risotto',
 'cook_minutes': None,
 'cook_time_category': 'inconnu',
 'nutri_score': 'E',
 'image_url': 'http://tastykitchen.com/recipes/wp-content/uploads/sites/2/2015/10/beet_corn_rissoto_tk-410x273.jpg',
 'ingredients_raw': ['3 whole Fresh Beets',
  '3 Tablespoons Olive Oil, Divided',
  '3 ears Corn, Kernels Removed (or About 2 Cups)',
  '5- 1/2 cups Chicken Stock',
  '2 Tablespoons Butter',
  '1/2 whole Onion',
  '2 cloves Garlic',
  '1- 1/2 cup Arborio Rice',
  '1 teaspoon Dried Thyme',
  '2 Tablespoons Lemon Juice, Plus Lemon Slices For Garnish',
  '2 teaspoons Salt',
  '1 teaspoon Ground Pepper',
  '1/2 cups Parmesan Cheese, Grated'],
 'instructions_text': 'Preheat oven to 400 degrees F. Cover a baking sheet with a piece of parchment paper. | Peel the beets with a vegetable peeler, cut into chunks, and place on the parchment-lined baking sheet. | Toss with 1 tablespoon of the olive oil. | Roast beets for 20 minutes, then remove fr

# 📊 8. Benchmark

Comparez les temps de réponse avant/après refactoring.

In [10]:
import time

tests = [
    ('search_by_name',       lambda: search_by_name('pasta')),
    ('search_by_ingredient', lambda: search_by_ingredient('tomato')),
    ('search_advanced',      lambda: search_advanced(max_time=30, nutri_score='A')),
    ('show_recipe',          lambda: show_recipe(first_id)),
]

print(f"{'Fonction':<25} {'Temps (ms)':>10}")
print('-' * 37)
for name, fn in tests:
    start = time.perf_counter()
    fn()
    elapsed = (time.perf_counter() - start) * 1000
    print(f"{name:<25} {elapsed:>9.1f} ms")

Fonction                  Temps (ms)
-------------------------------------
search_by_name               1261.1 ms
search_by_ingredient         2425.5 ms
search_advanced                 3.6 ms
🍽️  Beet & Corn Risotto
⏱️  Temps      : None min  [ inconnu ]
🔥 Calories   : None kcal
🥗 Nutri-score: E
🖼️  Image      : http://tastykitchen.com/recipes/wp-content/uploads/sites/2/2015/10/beet_corn_rissoto_tk-410x273.jpg

📝 Ingrédients:
['3 whole Fresh Beets', '3 Tablespoons Olive Oil, Divided', '3 ears Corn, Kernels Removed (or About 2 Cups)', '5- 1/2 cups Chicken Stock', '2 Tablespoons Butter', '1/2 whole Onion', '2 cloves Garlic', '1- 1/2 cup Arborio Rice', '1 teaspoon Dried Thyme', '2 Tablespoons Lemon Juice, Plus Lemon Slices For Garnish', '2 teaspoons Salt', '1 teaspoon Ground Pepper', '1/2 cups Parmesan Cheese, Grated']

👨‍🍳 Instructions:
Preheat oven to 400 degrees F. Cover a baking sheet with a piece of parchment paper. | Peel the beets with a vegetable peeler, cut into chunks, and place